In [ ]:
!pip install -q opendatasets
import opendatasets as od

od.download("https://www.kaggle.com/datasets/sonu1607/tosdr-terms-of-service-corpus")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: aadityatiwary2004
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/sonu1607/tosdr-terms-of-service-corpus


100%|██████████| 63.8M/63.8M [00:01<00:00, 37.0MB/s]


In [ ]:
import os
import pandas as pd

base_path = "/content/tosdr-terms-of-service-corpus"

txt_files = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".txt"):
            txt_files.append(os.path.join(root, file))

print("Total txt files found:", len(txt_files))
print("\nSample files:")
for f in txt_files[:10]:
    print(f)

Total txt files found: 9491

Sample files:
/content/tosdr-terms-of-service-corpus/text/Zenkit_Privacy.txt
/content/tosdr-terms-of-service-corpus/text/Canny_SecurityPolicy.txt
/content/tosdr-terms-of-service-corpus/text/Fossbytes_CookiePolicy.txt
/content/tosdr-terms-of-service-corpus/text/Kumu_TermsofUse.txt
/content/tosdr-terms-of-service-corpus/text/DentalWebsites_PrivacyPolicy.txt
/content/tosdr-terms-of-service-corpus/text/DigitalOcean_PrivacyPolicy.txt
/content/tosdr-terms-of-service-corpus/text/Telegram_FAQ.txt
/content/tosdr-terms-of-service-corpus/text/TicketTool_TermsofService.txt
/content/tosdr-terms-of-service-corpus/text/Factorio_TERMSOFSERVICE.txt
/content/tosdr-terms-of-service-corpus/text/Briar_Governance.txt


In [ ]:
data = []

for file_path in txt_files:
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read().strip()
            if len(text) > 50:
                data.append({
                    "filename": os.path.basename(file_path),
                    "text": text
                })
    except:
        pass

df = pd.DataFrame(data)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (9491, 2)


,filename,text
0,Zenkit_Privacy.txt,Web Privacy Policy\nThis privacy policy applie...
1,Canny_SecurityPolicy.txt,Log inSecurityWe take the security and privacy...
2,Fossbytes_CookiePolicy.txt,Cookie Policy for FossbytesThis is the Cookie ...
3,Kumu_TermsofUse.txt,Home \n\n\nTerms of Use\n\nLast updated May 11...
4,DentalWebsites_PrivacyPolicy.txt,Privacy Policy Web Site Terms and Conditions o...


In [ ]:
import re

clauses = []

for _, row in df.iterrows():
    filename = row["filename"]
    text = row["text"]

    parts = re.split(r'[\n\.!?;]+', text)

    for part in parts:
        part = part.strip()
        if len(part) > 30:
            clauses.append({
                "filename": filename,
                "clause": part
            })

clause_df = pd.DataFrame(clauses)

print("Total clauses:", clause_df.shape[0])
clause_df.head()

Total clauses: 1346919


,filename,clause
0,Zenkit_Privacy.txt,This privacy policy applies to the use of the ...
1,Zenkit_Privacy.txt,The privacy policy for the Zenkit software can...
2,Zenkit_Privacy.txt,Privacy Policy of Zenkit Website
3,Zenkit_Privacy.txt,Zenkit Website collects some Personal Data fro...
4,Zenkit_Privacy.txt,This document can be printed for reference by ...


In [ ]:
high_risk_keywords = [
    "share your data", "third party", "sell your data", "advertising",
    "without notice", "without consent", "track", "tracking", "collect location",
    "binding arbitration", "waive", "terminate", "suspend", "automatic renewal",
    "change these terms", "sole discretion", "not liable", "liability",
    "indemnify", "personal information", "cookies", "surveillance"
]

medium_risk_keywords = [
    "may update", "may modify", "reserve the right", "contact you",
    "policy changes", "service changes", "data retention", "account information",
    "usage information", "analytics", "notifications"
]

low_risk_keywords = [
    "you may delete", "you can request deletion", "opt out", "privacy rights",
    "your consent", "we protect", "you control", "data portability",
    "request removal", "account deletion", "user rights", "access your data"
]

def assign_risk(text):
    t = text.lower()

    for kw in high_risk_keywords:
        if kw in t:
            return 2  # High Risk

    for kw in medium_risk_keywords:
        if kw in t:
            return 1  # Medium Risk

    for kw in low_risk_keywords:
        if kw in t:
            return 0  # Low Risk

    return 1  # default Medium Risk

clause_df["risk"] = clause_df["clause"].apply(assign_risk)

clause_df["risk"].value_counts()

,count
risk,
1,1086621
2,245407
0,14891


In [ ]:
label_names = {0: "Low Risk", 1: "Medium Risk", 2: "High Risk"}

sample_df = clause_df.copy()
sample_df["risk_name"] = sample_df["risk"].map(label_names)

sample_df[["clause", "risk_name"]].head(20)

,clause,risk_name
0,This privacy policy applies to the use of the ...,Medium Risk
1,The privacy policy for the Zenkit software can...,Medium Risk
2,Privacy Policy of Zenkit Website,Medium Risk
3,Zenkit Website collects some Personal Data fro...,Medium Risk
4,This document can be printed for reference by ...,Medium Risk
5,Policy summary Personal Data collected for the...,Medium Risk
6,Facebook Ads conversion tracking (Facebook pix...,High Risk
7,Contact form and Mailing list or newsletter Pe...,Medium Risk
8,Displaying content from external platforms,Medium Risk
9,YouTube video widget Personal Data: Cookies,High Risk


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    clause_df["clause"],
    clause_df["risk"],
    test_size=0.2,
    random_state=42,
    stratify=clause_df["risk"]
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(
    y_test,
    y_pred,
    target_names=["Low Risk", "Medium Risk", "High Risk"]
))

Accuracy: 0.9713902830160663

              precision    recall  f1-score   support

    Low Risk       0.74      0.52      0.61      2978
 Medium Risk       0.98      0.99      0.98    217324
   High Risk       0.96      0.92      0.94     49082

    accuracy                           0.97    269384
   macro avg       0.89      0.81      0.84    269384
weighted avg       0.97      0.97      0.97    269384



In [ ]:
import pickle

bundle = {
    "model": model,
    "vectorizer": vectorizer
}

with open("ethical_risk_bundle.pkl", "wb") as f:
    pickle.dump(bundle, f)

print("Single model file saved successfully.")

Single model file saved successfully.


In [ ]:
from google.colab import files

files.download("ethical_risk_bundle.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pickle

with open("ethical_risk_bundle.pkl", "rb") as f:
    bundle = pickle.load(f)

loaded_model = bundle["model"]
loaded_vectorizer = bundle["vectorizer"]

print("Bundle loaded successfully.")

Bundle loaded successfully.


In [ ]:
def predict_risk(text):
    vec = loaded_vectorizer.transform([text])
    pred = loaded_model.predict(vec)[0]

    if pred == 0:
        return "Low Risk"
    elif pred == 1:
        return "Medium Risk"
    else:
        return "High Risk"

In [ ]:
print(predict_risk("We may share your personal information with third-party advertisers."))
print(predict_risk("You may request deletion of your account at any time."))
print(predict_risk("We reserve the right to modify these terms without notice."))

High Risk
Medium Risk
Medium Risk


In [ ]:
print(predict_risk("We share any personal information about your identity."))

High Risk
